# Módulo 4: Aprendizaje Automático (Machine Learning)
## Semana 1 - Clase 3: Regresión Lineal y Evaluación de Modelos

Este documento tiene como propósito introducir el primer modelo de aprendizaje supervisado continuo: **La Regresión Lineal**. Construiremos un sistema predictivo para estimar los cargos de seguros médicos utilizando datos reales de pacientes, conectando integralmente con el procesamiento matricial aprendido en la sesión anterior.

### **Objetivos de Aprendizaje:**
1. Integrar las técnicas de preprocesamiento (One-Hot Encoding, imputación y escalado) sobre un repositorio de datos en línea.
2. Demostrar la relación matemática subyacente entre variables mediante análisis de correlación y visualización de dispersión.
3. Entrenar y evaluar un modelo de Regresión Lineal Simple aislando una sola variable.
4. Escalar la arquitectura a una Regresión Lineal Múltiple guiada y cuantificar el incremento del poder predictivo.
5. Interpretar algorítmicamente el peso de cada variable analizando los coeficientes matemáticos de la ecuación (hiperplano).

---
## **Sección 1: Extracción de Datos y Preprocesamiento (Integración)**

### **Carga de Datos desde Repositorio**
Procederemos a extraer la información tabular de forma directa a través de un enlace *raw* desde GitHub hacia un objeto DataFrame de Pandas.

In [ ]:
import pandas as pd
import numpy as np

# Extracción directa del dataset público de Seguros Médicos
url_dataset = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'
df_seguros = pd.read_csv(url_dataset)

# Simulación de degradación de datos: Inyectaremos valores nulos (NaN) en la variable 'bmi' (IMC)
np.random.seed(42)
idx_nulos = np.random.choice(df_seguros.index, size=50, replace=False)
df_seguros.loc[idx_nulos, 'bmi'] = np.nan

display(df_seguros.head())

### **Requerimiento Práctico 1: Preprocesamiento Integral**
Antes de aplicar el modelo matemático, la matriz debe estar saneada. Se requiere ejecutar las siguientes rutinas de procesamiento de la sesión anterior:
1. Imputar los valores nulos en la columna `bmi` utilizando la mediana.
2. Aplicar *One-Hot Encoding* sobre las variables categóricas (`sex`, `smoker`, `region`). Asegúrese de utilizar el parámetro `drop_first=True` para evitar la colinealidad perfecta (trampa de variables dummy).
3. Verificar la ausencia de nulos e imprimir las dimensiones de la nueva matriz tabular (`df_final`).

In [ ]:
# TODO: Escribir el código para imputación y codificación (One-Hot Encoding)
# Sugerencia didáctica: Imprima los datos antes y después para visualizar el cambio.
# indices_nulos = df_seguros[df_seguros['bmi'].isnull()].index
# mediana_bmi = ...
# df_seguros['bmi'] = ...
# df_final = pd.get_dummies(...)


---
## **Sección 2: Análisis Exploratorio y Correlación**

### **Relación Matemática entre Variables**
La Regresión Lineal asume matemáticamente la existencia de una relación lineal (directa o inversamente proporcional) entre la variable independiente ($X$) y la variable objetivo ($y$). Para verificar esta premisa teórica, empleamos el **Coeficiente de Correlación de Pearson** ($r$) y diagramas de dispersión.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Omitimos advertencias visuales de la librería para un entorno limpio
import warnings
warnings.filterwarnings('ignore')

# Nota: Para este bloque de visualización, asumiremos que df_final contiene los datos ya preprocesados.
if 'df_final' not in locals():
    df_final = df_seguros.dropna().copy()

# Gráfico de dispersión: Edad vs Cargos de Seguro
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_final, x='age', y='charges', alpha=0.6, color='blue')
plt.title('Dispersión de Edad vs. Costos de Seguro')
plt.xlabel('Edad del Paciente')
plt.ylabel('Cargos (USD)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# Cálculo directo de la correlación de Pearson (r) entre edad y cargos
print("Correlación de Pearson (Edad vs Cargos):", df_final['age'].corr(df_final['charges']))

**Análisis de la Salida:** Existe una correlación positiva moderada ($\sim 0.30$). A medida que el paciente es de mayor edad, se observa un incremento general en los costos médicos. Las agrupaciones escalonadas y separadas en la gráfica insinúan la existencia de otras características determinantes (como el estatus de fumador).

---
## **Sección 3: Regresión Lineal Simple**

### **Mínimos Cuadrados Ordinarios (OLS)**
El algoritmo buscará trazar la "línea de mejor ajuste" ($y = mx + b$). Esta línea se calcula minimizando de forma matemática la suma de los residuos al cuadrado (la distancia geométrica entre la predicción teórica y el valor real de los datos).

Construiremos un modelo iterativo que intente predecir el **Cargo de Seguro** basándose única y exclusivamente en la **Edad**.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# 1. Definición de la variable independiente (X) en formato de matriz bidimensional (2D) 
# y la dependiente (y) en formato vectorial unidimensional (1D)
X_simple = df_final[['age']]
y = df_final['charges']

# 2. División estricta en subconjuntos de Entrenamiento (80%) y Prueba (20%)
X_train_simp, X_test_simp, y_train, y_test = train_test_split(X_simple, y, test_size=0.20, random_state=123)

# 3. Instanciación y Entrenamiento (Fitting) del Modelo Lineal
modelo_simple = LinearRegression()
modelo_simple.fit(X_train_simp, y_train)

# 4. Generación de matriz de predicciones utilizando el conjunto ciego de prueba (Test)
predicciones_simple = modelo_simple.predict(X_test_simp)

# 5. Visualización Matemática de la Línea de Ajuste
plt.figure(figsize=(10, 6))
plt.scatter(X_test_simp, y_test, color='blue', alpha=0.5, label='Datos Reales (Prueba)')
plt.plot(X_test_simp, predicciones_simple, color='red', linewidth=3, label='Modelo Lineal Predictivo')
plt.title('Regresión Lineal Simple: Predicción de Costos por Edad')
plt.xlabel('Edad')
plt.ylabel('Cargos Predichos (USD)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

---
## **Sección 4: Evaluación Matemática del Modelo**

Para auditar la validez y precisión teórica del modelo predictivo, recurrimos a las métricas de error estándar comparando los valores reales (`y_test`) contra nuestras proyecciones (`predicciones_simple`).
*   **MAE (Error Absoluto Medio):** Sumatoria del promedio del error en magnitud absoluta. Extremadamente fácil de interpretar en la misma escala que $y$.
*   **MSE (Error Cuadrático Medio):** Castiga de manera severa las predicciones con desviaciones grandes elevando la distancia residual al cuadrado.
*   **RMSE (Raíz del MSE):** Regresa la penalidad cuadrática a la unidad de medida original del problema de estudio.
*   **$R^2$ (Coeficiente de Determinación):** Métrica porcentual (de 0.0 a 1.0) que cuantifica el porcentaje exacto de la varianza en la variable objetivo explicada por la regresión del modelo.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_simp = mean_absolute_error(y_test, predicciones_simple)
mse_simp = mean_squared_error(y_test, predicciones_simple)
rmse_simp = np.sqrt(mse_simp)
r2_simp = r2_score(y_test, predicciones_simple)

print("--- DESEMPEÑO DEL MODELO LINEAL SIMPLE ---")
print(f"MAE:  ${mae_simp:,.2f}")
print(f"RMSE: ${rmse_simp:,.2f}")
print(f"R²:   {r2_simp:.4f}")

**Análisis Analítico:** Un $R^2$ matemáticamente cercano a $0.09$ indica que predecir el costo del seguro médico utilizando **única y exclusivamente la variable edad** origina un modelo altamente impreciso. De hecho, acarrea un margen de error residual promedio de casi \$9,300 (MAE). La arquitectura algorítmica requiere un escalamiento dimensional.

---
## **Sección 5: Regresión Lineal Múltiple e Interpretación Analítica**

Al introducir características matriciales suplementarias (e.g., IMC, Estatus de Fumador), la ecuación vectorial de la línea transiciona matemáticamente hacia un hiperplano multidimensional:
$y = \alpha + \beta_1(X_1) + \beta_2(X_2) + ... + \beta_n(X_n)$

### **Paso a Paso: Arquitectura de Modelo Múltiple**
En lugar de que lo resuelva usted mismo, exploraremos juntos en este bloque de código cómo se construye el modelo múltiple escalado:
1. Crearemos una nueva matriz predictora `X_mult` usando todas las columnas excepto `charges`.
2. Estandarizaremos las columnas numéricas (`age`, `bmi`, `children`) usando `StandardScaler`.
3. Entrenaremos el modelo múltiple y evaluaremos cómo mejora su $R^2$.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Definición de matriz independiente y vector objetivo
X_mult = df_final.drop(columns=['charges'])
y = df_final['charges']

# 2. Estandarización directa (solo sobre columnas numéricas continuas originales)
cols_escalar = ['age', 'bmi', 'children']
scaler = StandardScaler()
X_mult[cols_escalar] = scaler.fit_transform(X_mult[cols_escalar])

# 3. División estricta Entrenamiento / Prueba
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_mult, y, test_size=0.20, random_state=123)

# 4. Instanciación y Entrenamiento del Hiperplano Múltiple
modelo_multiple = LinearRegression()
modelo_multiple.fit(X_train_m, y_train_m)

# 5. Evaluación de Desempeño
predicciones_mult = modelo_multiple.predict(X_test_m)
r2_mult = r2_score(y_test_m, predicciones_mult)
rmse_mult = np.sqrt(mean_squared_error(y_test_m, predicciones_mult))

print(f"R² Múltiple: {r2_mult:.4f}")
print(f"RMSE Múltiple: ${rmse_mult:,.2f}")

### **Paso a Paso: Validación Visual (Realidad vs. Predicción)**

**La Analogía de la "Nota Final":**
Imagina que quieres adivinar la nota de un estudiante universitario. Si solo mides sus *horas de estudio* (1 variable), puedes dibujar una línea recta en una pizarra (como hicimos con la edad en la Sección 3). Pero si tu modelo usa *horas de estudio, horas de sueño, nivel de estrés y asistencia* (4 variables), ¡ya no puedes dibujarlo, porque los humanos no podemos ver un plano en 4 o 5 dimensiones!

Entonces, ¿cómo sabemos si nuestra ecuación invisible funciona? 
Para probarlo, graficamos una **"competencia"**. El eje $X$ será el Precio Real que la aseguradora pagó (la nota real del estudiante) y el eje $Y$ será el Precio Predicho (nuestra adivinanza). Si somos adivinos perfectos, todos los puntos formarán una línea diagonal exacta a 45 grados ($y = x$).

Observe el siguiente bloque de código para visualizar esta comprobación.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
# 1. Dibujar los puntos (Realidad vs Predicción)
sns.scatterplot(x=y_test_m, y=predicciones_mult, alpha=0.6, color='purple')

# 2. Dibujar la línea de "Perfección" a 45 grados (y = x)
max_val = max(y_test_m.max(), predicciones_mult.max())
plt.plot([0, max_val], [0, max_val], color='black', linestyle='--', linewidth=2, label='Predicción Perfecta (y = x)')

plt.title('Validación Visual: Costo Real vs. Costo Predicho')
plt.xlabel('Costo Médico REAL (y_test_m)')
plt.ylabel('Costo Médico PREDICHO (predicciones_mult)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

### **Paso a Paso: Inferencia Algorítmica (Coeficientes)**
El verdadero poder analítico de un modelo lineal múltiple no yace puramente en su exactitud predictiva general, sino en su transparencia de **inferencia algorítmica** de las variables.

Al extraer los coeficientes ponderados del hiperplano resultante, podemos inferir empíricamente qué variables tienen mayor impacto económico en los costos médicos. Ejecute la celda para imprimir el coeficiente asociado a cada característica.

In [ ]:
# Iteración sobre los nombres de columnas (X_mult) y los coeficientes del modelo
print("--- COEFICIENTES DEL HIPERPLANO DE REGRESIÓN ---")
for feature, coef in zip(X_mult.columns, modelo_multiple.coef_):
    print(f"{feature}: ${coef:,.2f}")

### **Reporte de Aprendizaje Final (Evaluación Teórica)**
Apoyándose en los resultados analíticos mostrados en los bloques anteriores, haga doble clic sobre este panel e introduzca sus respuestas teóricas justificadas para consolidar su aprendizaje:

**1. Discusión Métrica Computacional:** ¿Cómo cambió el coeficiente de determinación ($R^2$) al transicionar al hiperplano múltiple en contraste con el modelo de edad única? ¿Qué infiere este delta estadísticamente respecto al riesgo de negocio?
> *[Escriba su respuesta técnica argumentativa aquí...]*

**2. Inferencia Analítica sobre Factores Determinantes:** Observe detenidamente el coeficiente matemático asignado a la variable dummy representante de un paciente fumador (`smoker_yes`). ¿Qué valor numérico monetario exacto le asignó el algoritmo y qué significa esto teóricamente en una predicción comparando un paciente fumador frente a uno no fumador bajo las mismas condiciones fisiológicas (Edad e IMC)?
> *[Escriba su respuesta técnica argumentativa aquí...]*
